# Fine-tune Qwen3.5-0.8B làm Guardrail / Lesson-Scope Router (v13 full-train reviewed metrics-fix)

Notebook v11 này giữ pipeline **text-only SFT** cho router JSON, nhưng sửa phần môi trường theo kiểu **adaptive** cho GPU Blackwell / CUDA 13.

Mục tiêu model: đọc `input` của router và trả về **một JSON object hợp lệ** đúng 5 key:

```json
{
  "safety_label": "SAFE | UNSAFE | JAILBREAK",
  "topic_label": "ON_TOPIC | OFF_TOPIC | AMBIGUOUS | N_A",
  "action": "ALLOW_LESSON_ANSWER | SOFT_REFUSE_REDIRECT | ASK_CLARIFY | SAFETY_REFUSE",
  "attack_type": "none | prompt_injection | role_override | obfuscation | jailbreak_template | multilingual_jailbreak | unknown",
  "selected_kp_ids": []
}
```

Thiết kế chính:
- **Không ép `torch==2.8.0/cu128`** nữa. Official Colab dùng stack đó, nhưng RTX PRO 6000 Blackwell thường cần stack mới hơn như CUDA 13.0.
- Vẫn bám các nguyên tắc của guide Unsloth Qwen3.5: dùng Transformers v5, bf16/16-bit LoRA, không dùng QLoRA 4-bit.
- **text-only SFT**, không dùng `FastVisionModel` / `UnslothVisionDataCollator` vì dataset router không có ảnh.
- Dataset tải bằng `gdown` vào `/content` cho nhanh; output/checkpoint/adapter/eval lưu vào Google Drive.
- Có eval riêng cho router: JSON parse rate, exact match, field accuracy/F1, false-allow safety rate.


**v13 fixes:** robust JSONL loader for heterogeneous metadata, full train/eval by default, strict 5-key target validation, and safer TRL/Unsloth API compatibility wrappers.



## 0. Cài thư viện

Mặc định `INSTALL_PACKAGES=False` để không phá stack đang chạy. Nếu dùng runtime mới hoàn toàn, đổi thành `True`, chạy cell này một lần, restart runtime, rồi đổi lại `False`. Bản v9 **không downgrade/force reinstall torch stack** nếu runtime đã có PyTorch GPU mới, để tránh lỗi `torch +cu130` bị trộn với `torchvision +cu128`.

Sau khi chạy install, **restart runtime**, rồi chạy lại từ cell version check trở xuống.

Ghi chú:
- Với RTX PRO 6000 Blackwell, nếu runtime đã có `torch ... +cu130` và CUDA available thì giữ stack đó.
- Nếu chưa có torch, cell sẽ cài torch/torchvision/torchaudio từ CUDA 13 index.
- `flash-linear-attention` / `causal_conv1d` chỉ thử cài ở chế độ auto khi torch phù hợp; nếu không, notebook dùng fallback kernels.



In [ ]:
INSTALL_PACKAGES = False  # Current Blackwell runtime already has the stack; set True only on a fresh runtime
PRESERVE_EXISTING_TORCH_STACK = True
INSTALL_FAST_LINEAR_ATTENTION_KERNELS = "auto"  # True / False / "auto". Auto skips on newer torch where causal_conv1d often fails.
FAST_KERNELS_FAIL_HARD = False

if INSTALL_PACKAGES:
    import sys, subprocess, os, importlib.util
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "uv"])

    UV_BASE = [sys.executable, "-m", "uv", "pip"]
    UV_PIP = UV_BASE + ["install", "--python", sys.executable]
    UV_UNINSTALL = UV_BASE + ["uninstall", "--python", sys.executable]

    # 1) Torch stack policy.
    # Do NOT force torch==2.8/cu128 on Blackwell runtimes that already have cu130.
    torch_ok = False
    try:
        import torch
        torch_ok = torch.cuda.is_available()
        print("Existing torch:", torch.__version__, "CUDA:", torch.version.cuda, "CUDA available:", torch.cuda.is_available())
        if torch.cuda.is_available():
            print("GPU:", torch.cuda.get_device_name(0))
    except Exception as e:
        print("Torch not importable yet:", repr(e))

    if not torch_ok:
        print("Installing PyTorch CUDA 13.0 stack because torch CUDA is not available...")
        subprocess.check_call(
            UV_PIP + [
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu130",
            ]
        )
    else:
        print("Keeping existing torch stack. Not forcing cu128/cu130 reinstall.")

    # 2) Core Unsloth/Qwen3.5 deps.
    # Keep this broad to match Unsloth's current resolver on newer Blackwell runtimes.
    # Use Transformers v5+ for Qwen3.5.
    subprocess.check_call(UV_PIP + [
        "-U",
        "unsloth", "unsloth_zoo",
        "transformers>=5.0.0",
        "trl>=0.22.2",
        "datasets", "accelerate", "peft",
        "bitsandbytes", "torchao>=0.16.0",
        "pandas", "orjson", "tqdm", "matplotlib", "gdown",
    ])

    # sentence-transformers is unnecessary for this text-only router and can trigger torchcodec/FFmpeg import issues.
    subprocess.run(UV_UNINSTALL + ["-y", "sentence-transformers"], check=False)

    # 3) Optional acceleration kernels.
    # Official Colab/Qwen3.5 examples mention these, but causal_conv1d can fail on newer torch.
    should_try_fast_kernels = False
    if INSTALL_FAST_LINEAR_ATTENTION_KERNELS is True:
        should_try_fast_kernels = True
    elif INSTALL_FAST_LINEAR_ATTENTION_KERNELS == "auto":
        try:
            import torch
            # Conservative: official notebook comment says causal_conv1d path is tied to torch 2.8.
            should_try_fast_kernels = torch.__version__.startswith("2.8.")
        except Exception:
            should_try_fast_kernels = False

    if should_try_fast_kernels:
        fast_kernel_cmd = UV_PIP + [
            "--no-build-isolation",
            "flash-linear-attention",
            "causal_conv1d==1.6.0",
        ]
        try:
            subprocess.check_call(fast_kernel_cmd)
            print("Fast linear-attention kernels installed: flash-linear-attention + causal_conv1d==1.6.0")
        except subprocess.CalledProcessError as e:
            msg = (
                "WARNING: Could not install flash-linear-attention / causal_conv1d. "
                "Training will continue with fallback kernels. Set FAST_KERNELS_FAIL_HARD=True "
                "if you want this to stop the notebook."
            )
            print(msg)
            if FAST_KERNELS_FAIL_HARD:
                raise e
    else:
        print("Skipping flash-linear-attention / causal_conv1d install for this torch stack. Fallback kernels will be used.")

    print("Install done. IMPORTANT: restart runtime now, then run from the version-check cell downward.")


In [ ]:
# Optional sanity check after restarting runtime.
import importlib.metadata as importlib_metadata
from packaging.version import Version

CHECK_PACKAGES = [
    "torch", "torchvision", "torchaudio", "triton", "xformers", "bitsandbytes",
    "torchcodec", "torchao", "unsloth", "unsloth_zoo", "trl", "transformers",
    "flash-linear-attention", "causal-conv1d", "sentence-transformers",
]
for pkg in CHECK_PACKAGES:
    try:
        print(f"{pkg}: {importlib_metadata.version(pkg)}")
    except importlib_metadata.PackageNotFoundError:
        print(f"{pkg}: not installed")

import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
    try:
        print("arch list:", torch.cuda.get_arch_list())
    except Exception as e:
        print("arch list unavailable:", repr(e))

assert torch.cuda.is_available(), "CUDA is not available. This notebook expects a GPU runtime."
assert torch.cuda.is_bf16_supported(), "This run expects bf16 support. If false, reduce batch and allow fp16."

# For Qwen3.5, Unsloth docs require Transformers v5+.
try:
    transformers_version = Version(importlib_metadata.version("transformers"))
    assert transformers_version.major >= 5, f"Expected transformers v5+, got {transformers_version}"
except Exception as e:
    raise RuntimeError(f"Transformers version check failed: {e}")

# Torch/torchvision sanity check: do not require exact version; just fail on actual import/CUDA mismatch.
try:
    import torchvision
    print("torchvision import: OK", torchvision.__version__)
except Exception as e:
    raise RuntimeError(
        "torchvision import failed, likely torch/torchvision CUDA mismatch. "
        "Use a clean runtime, then install a matching torch/torchvision stack. "
        f"Original error: {type(e).__name__}: {e}"
    )

# bitsandbytes is imported by current Unsloth even for 16-bit LoRA, so module should exist.
try:
    import bitsandbytes as bnb
    print("bitsandbytes import: OK", getattr(bnb, "__version__", "unknown"))
except Exception as e:
    raise RuntimeError(f"bitsandbytes import failed: {type(e).__name__}: {e}")

# Optional fast-kernel import checks. Missing/failed imports are not fatal; they only affect speed.
for mod in ["fla", "causal_conv1d"]:
    try:
        __import__(mod)
        print(f"{mod}: import OK")
    except Exception as e:
        print(f"{mod}: import not available ({type(e).__name__}: {e})")

# Early Unsloth import probe. This should pass before loading data/model.
from unsloth import FastLanguageModel
print("Unsloth FastLanguageModel import: OK")


## 0.1. Tải dataset bằng gdown vào `/content`

Dataset được tải và giải nén trong `/content` để load nhanh. Không lưu dataset vào Google Drive. Drive chỉ dùng để lưu checkpoints, adapter, merged model và eval outputs.



In [ ]:
# Dataset input: lưu ở /content, không lưu vào Drive.
!mkdir -p /content/guardrail_router_v1_download
!gdown 1mzowsvxhoNHaU59cCbprQkIS3X7Qc8VI -O /content/guardrail_router_v1_download/guardrail_router_v1_downloaded
!ls -lh /content/guardrail_router_v1_download


In [ ]:

from pathlib import Path
import zipfile, tarfile, shutil, json, os

DOWNLOAD_ROOT = Path("/content/guardrail_router_v1_download")
DOWNLOADED_FILE = DOWNLOAD_ROOT / "guardrail_router_v1_downloaded"
EXTRACT_ROOT = DOWNLOAD_ROOT / "extracted"
LOCAL_DATA_ROOT = Path("/content/data/guardrail_router/v1")
REQUIRED_FILES = ["train.jsonl", "validation.jsonl", "test.jsonl"]

def find_dataset_dir(root: Path):
    candidates = []
    for train_path in root.rglob("train.jsonl"):
        d = train_path.parent
        if all((d / name).exists() for name in REQUIRED_FILES):
            candidates.append(d)
    if not candidates:
        return None
    # Prefer explicit guardrail path if present.
    candidates = sorted(candidates, key=lambda x: ("guardrail_router" not in str(x), len(str(x))))
    return candidates[0]

if not DOWNLOADED_FILE.exists():
    raise FileNotFoundError(f"Không thấy file tải về: {DOWNLOADED_FILE}")

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

if zipfile.is_zipfile(DOWNLOADED_FILE):
    print("Detected zip archive; extracting...")
    with zipfile.ZipFile(DOWNLOADED_FILE) as zf:
        zf.extractall(EXTRACT_ROOT)
elif tarfile.is_tarfile(DOWNLOADED_FILE):
    print("Detected tar archive; extracting...")
    with tarfile.open(DOWNLOADED_FILE) as tf:
        tf.extractall(EXTRACT_ROOT)
else:
    # Handle case gdown downloads a single jsonl file or a folder-like export is unavailable.
    print("Downloaded file is not zip/tar. Will search download root directly.")

found = find_dataset_dir(EXTRACT_ROOT) or find_dataset_dir(DOWNLOAD_ROOT)
if found is None:
    # Fallback: maybe files already exist under /content/data/guardrail_router/v1 from a previous run.
    found = find_dataset_dir(Path("/content/data"))

if found is None:
    raise FileNotFoundError(
        "Không tìm thấy train.jsonl / validation.jsonl / test.jsonl sau khi tải gdown. "
        "Hãy kiểm tra Google Drive file có phải zip/folder export chứa dataset không."
    )

DATA_DIR = found
TRAIN_PATH = DATA_DIR / "train.jsonl"
VAL_PATH = DATA_DIR / "validation.jsonl"
TEST_PATH = DATA_DIR / "test.jsonl"
MANIFEST_PATH = DATA_DIR / "manifest.json"

print("DATA_DIR =", DATA_DIR)
for path in [TRAIN_PATH, VAL_PATH, TEST_PATH, MANIFEST_PATH]:
    print(path, "exists=", path.exists(), "size=", path.stat().st_size if path.exists() else None)

if str(DATA_DIR).startswith("/content/drive"):
    raise RuntimeError("DATA_DIR đang trỏ vào Drive. Dataset phải nằm ở /content để load nhanh.")


## 0.2. Mount Google Drive cho output

Chỉ output được lưu vào Drive: checkpoints, adapter, eval predictions, optional merged model.



In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 1. Import, seed, kiểm tra GPU



In [ ]:
import os, re, json, math, random, warnings, subprocess, inspect
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, Features, Value, Sequence, concatenate_datasets
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    try:
        print(subprocess.check_output(["nvidia-smi"], text=True)[:2000])
    except Exception as e:
        print("nvidia-smi unavailable:", e)


## 2. Cấu hình đường dẫn dataset

`DATA_DIR` đã được set từ cell `gdown` ở đầu notebook. Dataset nằm trong `/content`, không nằm trong Drive.



In [ ]:

# DATA_DIR đã được set ở cell gdown/extract phía trên.
# Fallback chỉ dùng local /content hoặc /mnt/data, tuyệt đối không tự tìm trong Drive để tránh load chậm.
if "DATA_DIR" not in globals():
    CANDIDATE_DATA_DIRS = [
        Path("/content/data/guardrail_router/v1"),
        Path("/mnt/data"),  # chỉ dùng khi validate trong môi trường artifact
        Path("."),
    ]
    DATA_DIR = None
    for d in CANDIDATE_DATA_DIRS:
        if (d / "train.jsonl").exists() and (d / "validation.jsonl").exists() and (d / "test.jsonl").exists():
            DATA_DIR = d
            break

if DATA_DIR is None:
    raise FileNotFoundError("Không tìm thấy train.jsonl / validation.jsonl / test.jsonl trong /content. Hãy chạy cell gdown ở đầu notebook.")

if str(DATA_DIR).startswith("/content/drive"):
    raise RuntimeError("DATA_DIR đang trỏ vào Drive. Dataset phải nằm ở /content để load nhanh.")

TRAIN_PATH = DATA_DIR / "train.jsonl"
VAL_PATH   = DATA_DIR / "validation.jsonl"
TEST_PATH  = DATA_DIR / "test.jsonl"
MANIFEST_PATH = DATA_DIR / "manifest.json"

print("DATA_DIR =", DATA_DIR)
print(TRAIN_PATH, TRAIN_PATH.stat().st_size)
print(VAL_PATH, VAL_PATH.stat().st_size)
print(TEST_PATH, TEST_PATH.stat().st_size)
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    print(json.dumps(manifest, ensure_ascii=False, indent=2)[:2500])


## 3. Load dataset và kiểm tra phân bố



In [ ]:
# Robust JSONL loader.
# Do NOT use datasets.load_dataset("json") here.
# Reason: `metadata` has heterogeneous keys across sources (EduVidQA, CLINC, WildGuardMix, etc.).
# Arrow infers a fixed struct schema from early rows, then fails when later rows have different metadata keys.
# Fix: keep `output` as a fixed 5-key struct and serialize heterogeneous `metadata` to a JSON string.

from datasets import Dataset, DatasetDict, Features, Value, Sequence

LOAD_TARGET_KEYS = ["safety_label", "topic_label", "action", "attack_type", "selected_kp_ids"]
LOAD_TARGET_KEY_SET = set(LOAD_TARGET_KEYS)

ROUTER_FEATURES = Features({
    "id": Value("string"),
    "source": Value("string"),
    "input": Value("string"),
    "output": {
        "safety_label": Value("string"),
        "topic_label": Value("string"),
        "action": Value("string"),
        "attack_type": Value("string"),
        "selected_kp_ids": Sequence(Value("string")),
    },
    # Heterogeneous source-specific metadata; parse with json.loads(row["metadata"]) when needed.
    "metadata": Value("string"),
})

def load_router_jsonl(path: Path) -> Dataset:
    rows = []
    bad_targets = []
    missing_required = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except Exception as e:
                bad_targets.append((line_no, f"json_decode_error:{type(e).__name__}", line[:200]))
                continue

            for k in ["id", "source", "input", "output"]:
                if k not in row:
                    missing_required.append((line_no, k))

            y = row.get("output")
            if not isinstance(y, dict):
                bad_targets.append((line_no, "output_not_dict", type(y).__name__))
                continue

            y_keys = set(y.keys())
            if y_keys != LOAD_TARGET_KEY_SET:
                missing = sorted(LOAD_TARGET_KEY_SET - y_keys)
                extra = sorted(y_keys - LOAD_TARGET_KEY_SET)
                bad_targets.append((line_no, f"target_keys_mismatch:missing={missing};extra={extra}", sorted(y_keys)))
                continue

            selected_kp_ids = y.get("selected_kp_ids")
            if selected_kp_ids is None:
                selected_kp_ids = []
            if not isinstance(selected_kp_ids, list):
                bad_targets.append((line_no, "selected_kp_ids_not_list", type(selected_kp_ids).__name__))
                continue

            metadata = row.get("metadata") or {}
            if not isinstance(metadata, dict):
                metadata = {"_raw_metadata": metadata}

            rows.append({
                "id": str(row.get("id", "")),
                "source": str(row.get("source", "")),
                "input": str(row.get("input", "")),
                "output": {
                    "safety_label": str(y["safety_label"]),
                    "topic_label": str(y["topic_label"]),
                    "action": str(y["action"]),
                    "attack_type": str(y["attack_type"]),
                    "selected_kp_ids": [str(x) for x in selected_kp_ids],
                },
                "metadata": json.dumps(metadata, ensure_ascii=False, separators=(",", ":")),
            })

    if missing_required:
        raise ValueError(f"{path} has rows missing required keys. First errors: {missing_required[:10]}")
    if bad_targets:
        preview = bad_targets[:10]
        raise ValueError(
            f"{path} has {len(bad_targets)} invalid rows. First errors: {preview}. "
            "Targets must contain exactly 5 keys and no extras."
        )
    if not rows:
        raise ValueError(f"{path} loaded 0 rows; check the file contents.")

    return Dataset.from_list(rows, features=ROUTER_FEATURES)

raw = DatasetDict({
    "train": load_router_jsonl(TRAIN_PATH),
    "validation": load_router_jsonl(VAL_PATH),
    "test": load_router_jsonl(TEST_PATH),
})

print(raw)
print("Rows:", {split: len(raw[split]) for split in raw})
print("Columns:", raw["train"].column_names)
print("Sample keys:", raw["train"][0].keys())
print("Sample output:", raw["train"][0]["output"])
print("Sample metadata JSON:", raw["train"][0]["metadata"][:300])

# Hard assertions for the known v1 dataset. If you intentionally use a different dataset, update these counts.
EXPECTED_SPLIT_COUNTS = {"train": 10490, "validation": 915, "test": 1445}
actual_counts = {split: len(raw[split]) for split in raw}
if actual_counts != EXPECTED_SPLIT_COUNTS:
    print("WARNING: split counts differ from original v1 manifest:", actual_counts, "expected:", EXPECTED_SPLIT_COUNTS)
else:
    print("Split counts match original v1 manifest.")


In [ ]:
def metadata_dict(row):
    md = row.get("metadata") or {}
    if isinstance(md, dict):
        return md
    if isinstance(md, str) and md.strip():
        try:
            return json.loads(md)
        except Exception:
            return {"_metadata_parse_error": md[:200]}
    return {}

def summarize_split(ds, name):
    rows = []
    for r in ds:
        y = r["output"]
        md = metadata_dict(r)
        rows.append({
            "split": name,
            "source": r.get("source", ""),
            "safety_label": y.get("safety_label"),
            "topic_label": y.get("topic_label"),
            "action": y.get("action"),
            "attack_type": y.get("attack_type"),
            "target": f'{y.get("safety_label")}::{y.get("topic_label")}',
            "route_group": md.get("route_group", ""),
        })
    return pd.DataFrame(rows)

summary_df = pd.concat([
    summarize_split(raw["train"], "train"),
    summarize_split(raw["validation"], "validation"),
    summarize_split(raw["test"], "test"),
], ignore_index=True)

for col in ["source", "target", "action", "attack_type", "route_group"]:
    print("\n==", col, "==")
    display(pd.crosstab(summary_df["split"], summary_df[col], margins=True))


## 4. Target schema + canonical JSON

Ta ép output thành JSON compact với thứ tự key cố định. Điều này giúp model học format ổn hơn và eval exact-match dễ hơn.



In [ ]:
ALLOWED = {
    "safety_label": {"SAFE", "UNSAFE", "JAILBREAK"},
    "topic_label": {"ON_TOPIC", "OFF_TOPIC", "AMBIGUOUS", "N_A"},
    "action": {"ALLOW_LESSON_ANSWER", "SOFT_REFUSE_REDIRECT", "ASK_CLARIFY", "SAFETY_REFUSE"},
    "attack_type": {
        "none", "prompt_injection", "role_override", "obfuscation",
        "jailbreak_template", "multilingual_jailbreak", "unknown",
    },
}
TARGET_KEYS = ["safety_label", "topic_label", "action", "attack_type", "selected_kp_ids"]
TARGET_KEY_SET = set(TARGET_KEYS)

ROUTER_SYSTEM_RULES = (
    "You are a lesson-scope safety router.\n"
    "Return exactly one valid JSON object and nothing else.\n"
    "Do not use markdown. Do not explain.\n"
    "The JSON object must contain exactly these 5 keys and no extra keys: "
    "safety_label, topic_label, action, attack_type, selected_kp_ids."
)

def canonical_target(y):
    obj = {
        "safety_label": y["safety_label"],
        "topic_label": y["topic_label"],
        "action": y["action"],
        "attack_type": y.get("attack_type", "none"),
        "selected_kp_ids": y.get("selected_kp_ids", []),
    }
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))

def validate_target_obj(obj, strict=True):
    if not isinstance(obj, dict):
        return False, "not_dict"

    if strict:
        actual_keys = set(obj.keys())
        if actual_keys != TARGET_KEY_SET:
            missing = sorted(TARGET_KEY_SET - actual_keys)
            extra = sorted(actual_keys - TARGET_KEY_SET)
            return False, f"target_keys_mismatch:missing={missing},extra={extra}"
    else:
        for k in TARGET_KEYS:
            if k not in obj:
                return False, f"missing_{k}"

    for k, vals in ALLOWED.items():
        if obj.get(k) not in vals:
            return False, f"bad_{k}:{obj.get(k)}"
    if not isinstance(obj.get("selected_kp_ids"), list):
        return False, "selected_kp_ids_not_list"
    if obj["safety_label"] in {"UNSAFE", "JAILBREAK"}:
        if obj["topic_label"] != "N_A" or obj["action"] != "SAFETY_REFUSE" or obj["selected_kp_ids"] != []:
            return False, "unsafe_jailbreak_invariant"
    return True, "ok"

bad = []
for split in raw:
    for i, r in enumerate(raw[split]):
        ok, reason = validate_target_obj(r["output"], strict=True)
        if not ok:
            bad.append((split, i, reason, r["output"]))
print("target violations:", len(bad))
if bad[:5]:
    print(bad[:5])


## 5. Load Qwen3.5-0.8B và convert sang chat-template text

Dataset gốc đã có `input` chứa task/scope/query. Ta thêm một system rule rất ngắn để khóa JSON-only behavior.



In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "unsloth/Qwen3.5-0.8B"
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = False
LOAD_IN_16BIT = True
FULL_FINETUNING = False

# Với GPU ~95GB VRAM: tắt gradient checkpointing để nhanh hơn.
# Nếu OOM hoặc dùng GPU nhỏ, đổi thành "unsloth".
USE_GRADIENT_CHECKPOINTING = False

from_pretrained_kwargs = dict(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    full_finetuning=FULL_FINETUNING,
)

# Current Unsloth often auto-switches to 16-bit LoRA when load_in_4bit=False and full_finetuning=False.
# Some versions accept load_in_16bit, some do not. Try it first, then fallback cleanly.
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        **from_pretrained_kwargs,
        load_in_16bit=LOAD_IN_16BIT,
    )
except TypeError as e:
    if "load_in_16bit" not in str(e):
        raise
    print("This Unsloth version does not accept load_in_16bit; falling back to load_in_4bit=False + full_finetuning=False.")
    model, tokenizer = FastLanguageModel.from_pretrained(**from_pretrained_kwargs)

# Unsloth/Qwen3.5 may return either a plain tokenizer or a multimodal processor.
# If it is a processor, calling processor(text_string) positionally can be interpreted as images=...
# and crash with "Incorrect image source". Keep the processor for saving if needed,
# but use the underlying text tokenizer for all text-only tokenization/training/eval.
processor = tokenizer
text_tokenizer = getattr(processor, "tokenizer", processor)

if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token
if hasattr(processor, "pad_token") and getattr(processor, "pad_token", None) is None:
    try:
        processor.pad_token = text_tokenizer.pad_token
    except Exception:
        pass

# Backward-compatible alias: from here onward, `tokenizer` means the text tokenizer, not the VL processor.
tokenizer = text_tokenizer

def apply_router_chat_template(messages, tokenize=False, add_generation_prompt=False):
    """Apply chat template using the text tokenizer, avoiding VL processor image handling."""
    return tokenizer.apply_chat_template(
        messages,
        tokenize=tokenize,
        add_generation_prompt=add_generation_prompt,
    )

def tokenize_text(text_or_texts, **kwargs):
    """Tokenize text safely. Always uses text= keyword if the object is processor-like."""
    return tokenizer(text_or_texts, **kwargs)

print("Loaded:", MODEL_NAME)
print("processor class:", type(processor).__name__)
print("text tokenizer class:", type(tokenizer).__name__)
print("pad_token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token, tokenizer.eos_token_id)


In [ ]:
def row_to_messages(row):
    user_content = ROUTER_SYSTEM_RULES + "\n\n" + row["input"].strip()
    assistant_content = canonical_target(row["output"])
    return [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]

def row_to_text(row):
    text = apply_router_chat_template(
        row_to_messages(row),
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_text = raw["train"].map(row_to_text, remove_columns=raw["train"].column_names, desc="format train")
val_text   = raw["validation"].map(row_to_text, remove_columns=raw["validation"].column_names, desc="format val")
test_text  = raw["test"].map(row_to_text, remove_columns=raw["test"].column_names, desc="format test")

print(train_text)
print(train_text[0]["text"][:1200])


## 6. Optional: oversample AMBIGUOUS / ASK_CLARIFY trong train

Vì `AMBIGUOUS` ít, bật oversample nhẹ giúp model không ép mọi câu mơ hồ thành `ON_TOPIC`/`OFF_TOPIC`.

Mặc định: bật `OVERSAMPLE_AMBIGUOUS=True`, repeat thêm 2 lần nhóm `action=ASK_CLARIFY`.



In [ ]:
OVERSAMPLE_AMBIGUOUS = True
AMBIGUOUS_EXTRA_REPEATS = 2

if OVERSAMPLE_AMBIGUOUS:
    ambiguous_indices = [
        i for i, r in enumerate(raw["train"])
        if r["output"].get("action") == "ASK_CLARIFY" or r["output"].get("topic_label") == "AMBIGUOUS"
    ]
    print("ambiguous train rows:", len(ambiguous_indices))
    if ambiguous_indices:
        amb_raw = raw["train"].select(ambiguous_indices)
        amb_text = amb_raw.map(row_to_text, remove_columns=amb_raw.column_names, desc="format ambiguous")
        train_text_final = concatenate_datasets([train_text] + [amb_text] * AMBIGUOUS_EXTRA_REPEATS).shuffle(seed=SEED)
    else:
        train_text_final = train_text
else:
    train_text_final = train_text

print("train before:", len(train_text))
print("train final:", len(train_text_final))


## 7. Token length diagnostics

Nếu nhiều sample bị truncate, tăng `MAX_SEQ_LENGTH`. Với router hiện tại, 4096 thường ổn.



In [ ]:
def token_length_stats(ds, n=None):
    if n is None or n > len(ds):
        n = len(ds)
    lengths = []
    for i in tqdm(range(n), desc="token length"):
        enc = tokenize_text(ds[i]["text"], add_special_tokens=False)
        lengths.append(len(enc["input_ids"]))
    s = pd.Series(lengths)
    return s.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame("tokens")

length_stats = token_length_stats(train_text_final, n=min(2000, len(train_text_final)))
display(length_stats)
print("MAX_SEQ_LENGTH =", MAX_SEQ_LENGTH)


## 8. Gắn LoRA adapter

Cấu hình mặc định cho router v1:

- `r=16`, `lora_alpha=16`
- attention + MLP projection modules
- `gradient_checkpointing=False` cho GPU lớn

Nếu underfit: thử `r=32`, `lora_alpha=32`, `learning_rate=7e-5`.



In [ ]:
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

peft_kwargs = dict(
    model=model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

try:
    model = FastLanguageModel.get_peft_model(**peft_kwargs)
except TypeError as e:
    if "max_seq_length" not in str(e):
        raise
    print("get_peft_model() does not accept max_seq_length in this Unsloth version; retrying without it.")
    peft_kwargs.pop("max_seq_length", None)
    model = FastLanguageModel.get_peft_model(**peft_kwargs)

model.print_trainable_parameters()


## 9. Train

Cấu hình dưới tối ưu cho GPU lớn. Nếu OOM, giảm batch về 32 hoặc bật gradient checkpointing.

Notebook dùng `optim="adamw_torch"` để tránh phụ thuộc 8-bit optimizer trong train; `bitsandbytes` vẫn được cài vì Unsloth import cần module này. Với TRL mới, dùng `max_length=MAX_SEQ_LENGTH` trong `SFTConfig`.



In [ ]:
RUN_NAME = "qwen35_08b_guardrail_router_lora_v13_text_tokenizer_fix"
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/qwen35_08b_guardrail_router_runs")
RUN_DIR = DRIVE_RUN_ROOT / RUN_NAME
OUTPUT_DIR = str(RUN_DIR / "trainer_checkpoints")
PRED_DIR = RUN_DIR / "eval_predictions"
ADAPTER_DIR = str(RUN_DIR / "adapter")
MERGED_DIR = str(RUN_DIR / "merged_16bit")

RUN_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_DIR =", RUN_DIR)
print("OUTPUT_DIR =", OUTPUT_DIR)

# FULL TRAIN: no max_steps path. Run all will train on the full train split.
PER_DEVICE_TRAIN_BATCH_SIZE = 64   # GPU 95GB: thử 64 trước, rồi 128 nếu còn dư VRAM
PER_DEVICE_EVAL_BATCH_SIZE = 64
GRAD_ACCUM_STEPS = 1
NUM_EPOCHS = 2
LEARNING_RATE = 5e-5

sft_config_kwargs = dict(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,  # TRL 0.22+ uses max_length
    packing=False,

    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,

    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    weight_decay=0.001,
    lr_scheduler_type="linear",
    optim="adamw_torch",  # 95GB VRAM: 8-bit optimizer not needed

    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    dataloader_num_workers=4,
    report_to="none",
    seed=SEED,
    dataset_num_proc=2,
)

def build_sft_config(kwargs):
    params = set(inspect.signature(SFTConfig).parameters)
    cfg = dict(kwargs)
    # Version compatibility: older args used max_seq_length / evaluation_strategy.
    if "max_length" in cfg and "max_length" not in params and "max_seq_length" in params:
        cfg["max_seq_length"] = cfg.pop("max_length")
    if "eval_strategy" in cfg and "eval_strategy" not in params and "evaluation_strategy" in params:
        cfg["evaluation_strategy"] = cfg.pop("eval_strategy")
    supported = {k: v for k, v in cfg.items() if k in params}
    dropped = sorted(set(cfg) - set(supported))
    if dropped:
        print("SFTConfig dropped unsupported keys for this TRL version:", dropped)
    return SFTConfig(**supported)

training_args = build_sft_config(sft_config_kwargs)

print("SFTTrainer will use text tokenizer:", type(tokenizer).__name__)

trainer_kwargs = dict(
    model=model,
    train_dataset=train_text_final,
    eval_dataset=val_text,
    args=training_args,
)
trainer_params = set(inspect.signature(SFTTrainer).parameters)
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer
else:
    print("SFTTrainer signature has neither processing_class nor tokenizer; relying on model/tokenizer defaults.")

trainer = SFTTrainer(**trainer_kwargs)

# Quan trọng cho router: chỉ tính loss trên phần assistant JSON, không học nhại prompt dài.
def detect_chat_markers(example_text):
    candidates = [
        ("<|im_start|>user\n", "<|im_start|>assistant\n"),
        ("<|im_start|>user", "<|im_start|>assistant"),
    ]
    for instruction_part, response_part in candidates:
        if instruction_part in example_text and response_part in example_text:
            return instruction_part, response_part
    raise ValueError(
        "Không dò được chat markers cho train_on_responses_only. "
        "Hãy print train_text_final[0]['text'][:1000] rồi chỉnh instruction_part/response_part."
    )

TRAIN_ON_RESPONSES_ONLY = True
if TRAIN_ON_RESPONSES_ONLY:
    from unsloth.chat_templates import train_on_responses_only
    instruction_part, response_part = detect_chat_markers(train_text_final[0]["text"])
    print("instruction_part =", repr(instruction_part))
    print("response_part =", repr(response_part))
    trainer = train_on_responses_only(
        trainer,
        instruction_part=instruction_part,
        response_part=response_part,
    )
    print("Enabled train_on_responses_only")

print("FULL TRAIN CONFIG")
print("train rows:", len(train_text_final), "base train rows:", len(train_text), "validation rows:", len(val_text))
print("epochs:", NUM_EPOCHS, "batch:", PER_DEVICE_TRAIN_BATCH_SIZE, "grad_accum:", GRAD_ACCUM_STEPS)
trainer.train()


## 10. Save LoRA adapter



In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
# If Unsloth returned a multimodal processor, save it under a separate folder for reproducibility.
try:
    if 'processor' in globals() and processor is not tokenizer:
        processor.save_pretrained(str(Path(ADAPTER_DIR) / "processor"))
except Exception as e:
    print("Warning: could not save processor separately:", repr(e))
print("Saved adapter and text tokenizer to", ADAPTER_DIR)


## 11. Inference helper: generate JSON-only



In [ ]:
FastLanguageModel.for_inference(model)

GEN_MAX_NEW_TOKENS = 128

def build_prompt(row_or_input):
    if isinstance(row_or_input, dict):
        inp = row_or_input["input"]
    else:
        inp = str(row_or_input)
    messages = [{"role": "user", "content": ROUTER_SYSTEM_RULES + "\n\n" + inp.strip()}]
    return apply_router_chat_template(messages, tokenize=False, add_generation_prompt=True)

def extract_json_substring(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    return text[start:end+1]

def parse_router_output(text):
    js = extract_json_substring(text)
    if js is None:
        return None, "no_json"
    try:
        obj = json.loads(js)
    except Exception as e:
        return None, f"json_error:{type(e).__name__}"
    ok, reason = validate_target_obj(obj)
    if not ok:
        return None, reason
    obj = {k: obj[k] for k in TARGET_KEYS}
    return obj, "ok"

@torch.no_grad()
def predict_one(row_or_input):
    prompt = build_prompt(row_or_input)
    inputs = tokenize_text(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    obj, status = parse_router_output(gen)
    return {"raw": gen, "parsed": obj, "status": status}

for i in range(3):
    r = raw["validation"][i]
    pred = predict_one(r)
    print("\n--- sample", i, "---")
    print("gold:", canonical_target(r["output"]))
    print("pred raw:", pred["raw"])
    print("status:", pred["status"])


## 12. Eval trên validation/test

Mặc định eval **full validation** và **full test** trong bản v11.



In [ ]:
MAX_EVAL_SAMPLES = None
assert MAX_EVAL_SAMPLES is None, "v11 is configured for full evaluation; keep MAX_EVAL_SAMPLES=None."

@torch.no_grad()
def predict_batch(rows, batch_size=32):
    results = []
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"  # safer for batched decoder-only generation
    try:
        for start in tqdm(range(0, len(rows), batch_size), desc="generate"):
            batch = rows[start:start+batch_size]
            prompts = [build_prompt(r) for r in batch]
            inputs = tokenize_text(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            ).to(model.device)
            prompt_width = inputs["input_ids"].shape[1]
            out = model.generate(
                **inputs,
                max_new_tokens=GEN_MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            for j in range(len(batch)):
                gen_ids = out[j][prompt_width:]
                gen = tokenizer.decode(gen_ids, skip_special_tokens=True)
                obj, status = parse_router_output(gen)
                results.append({"raw": gen, "parsed": obj, "status": status})
    finally:
        tokenizer.padding_side = old_padding_side
    return results

def normalize_kp_ids(ids):
    """Order-insensitive KP comparison helper.

    selected_kp_ids is semantically a set for router evaluation.
    Sorting + de-duplicating avoids marking ["a", "b"] vs ["b", "a"] as different.
    """
    if ids is None:
        return tuple()
    return tuple(sorted(set(map(str, ids))))

def flatten_target(obj):
    return {
        "safety_label": obj.get("safety_label"),
        "topic_label": obj.get("topic_label"),
        "action": obj.get("action"),
        "attack_type": obj.get("attack_type"),
        "selected_kp_ids": normalize_kp_ids(obj.get("selected_kp_ids", [])),
        "target": f'{obj.get("safety_label")}::{obj.get("topic_label")}',
    }

def simple_accuracy(y_true, y_pred):
    """Standard accuracy: number of exactly correct predictions divided by N."""
    y_true = list(map(str, y_true))
    y_pred = list(map(str, y_pred))
    if not y_true:
        return 0.0
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

def simple_macro_f1(y_true, y_pred):
    """Standard unweighted macro-F1.

    This matches sklearn.metrics.f1_score(y_true, y_pred, average="macro", zero_division=0)
    for the labels present in either gold or predictions.
    """
    y_true = list(map(str, y_true))
    y_pred = list(map(str, y_pred))
    labels = sorted(set(y_true) | set(y_pred))
    if not labels:
        return 0.0
    f1s = []
    for label in labels:
        tp = sum(t == label and p == label for t, p in zip(y_true, y_pred))
        fp = sum(t != label and p == label for t, p in zip(y_true, y_pred))
        fn = sum(t == label and p != label for t, p in zip(y_true, y_pred))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        f1s.append(f1)
    return sum(f1s) / len(f1s)

def evaluate_split(split_name="validation", batch_size=32):
    ds = raw[split_name]
    n = len(ds) if MAX_EVAL_SAMPLES is None else min(MAX_EVAL_SAMPLES, len(ds))
    rows = [ds[i] for i in range(n)]
    preds = predict_batch(rows, batch_size=batch_size)

    records = []
    for r, p in zip(rows, preds):
        gold = flatten_target(r["output"])
        pred_obj = p["parsed"]
        pred = flatten_target(pred_obj) if pred_obj is not None else {
            "safety_label": "__INVALID__",
            "topic_label": "__INVALID__",
            "action": "__INVALID__",
            "attack_type": "__INVALID__",
            "selected_kp_ids": tuple(),
            "target": "__INVALID__",
        }
        exact = pred_obj is not None and canonical_target(pred_obj) == canonical_target(r["output"])
        # Count KP exact-set accuracy only for schema-valid predictions.
        # Invalid JSON should not get a free pass just because gold selected_kp_ids is empty.
        selected_kp_exact = pred_obj is not None and gold["selected_kp_ids"] == pred["selected_kp_ids"]
        records.append({
            "id": r.get("id"),
            "source": r.get("source"),
            "status": p["status"],
            "raw_pred": p["raw"],
            "exact_match": exact,
            "selected_kp_exact_set": selected_kp_exact,
            **{f"gold_{k}": v for k, v in gold.items()},
            **{f"pred_{k}": v for k, v in pred.items()},
        })
    df = pd.DataFrame(records)

    metrics = {
        "split": split_name,
        "n": len(df),
        "valid_json_rate": float((df["status"] == "ok").mean()),
        "exact_match": float(df["exact_match"].mean()),
        "selected_kp_exact_set_accuracy": float(df["selected_kp_exact_set"].mean()),
    }

    for field in ["safety_label", "topic_label", "action", "attack_type", "target"]:
        y_true = df[f"gold_{field}"].astype(str).tolist()
        y_pred = df[f"pred_{field}"].astype(str).tolist()
        metrics[f"{field}_accuracy"] = float(simple_accuracy(y_true, y_pred))
        metrics[f"{field}_macro_f1"] = float(simple_macro_f1(y_true, y_pred))

    unsafe_mask = df["gold_safety_label"].isin(["UNSAFE", "JAILBREAK"])
    if unsafe_mask.any():
        metrics["unsafe_or_jailbreak_false_allow_rate"] = float((df.loc[unsafe_mask, "pred_safety_label"] == "SAFE").mean())
    safe_mask = df["gold_safety_label"].eq("SAFE")
    if safe_mask.any():
        metrics["safe_false_safety_refuse_rate"] = float((df.loc[safe_mask, "pred_action"] == "SAFETY_REFUSE").mean())
    ambiguous_mask = df["gold_topic_label"].eq("AMBIGUOUS")
    if ambiguous_mask.any():
        metrics["ambiguous_recall"] = float((df.loc[ambiguous_mask, "pred_topic_label"] == "AMBIGUOUS").mean())

    PRED_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(PRED_DIR / f"{split_name}_predictions.csv", index=False)
    with open(PRED_DIR / f"{split_name}_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    return metrics, df

val_metrics, val_pred_df = evaluate_split("validation", batch_size=32)
print(json.dumps(val_metrics, ensure_ascii=False, indent=2))
display(val_pred_df.head())


In [ ]:
test_metrics, test_pred_df = evaluate_split("test", batch_size=32)
print(json.dumps(test_metrics, ensure_ascii=False, indent=2))
display(test_pred_df.head())


## 13. Error analysis nhanh



In [ ]:
def show_error_analysis(df, name="validation", n=20):
    print("==", name, "status counts ==")
    display(df["status"].value_counts().to_frame("count"))

    print("\n== safety confusion ==")
    display(pd.crosstab(df["gold_safety_label"], df["pred_safety_label"], margins=True))

    print("\n== topic confusion ==")
    display(pd.crosstab(df["gold_topic_label"], df["pred_topic_label"], margins=True))

    print("\n== action confusion ==")
    display(pd.crosstab(df["gold_action"], df["pred_action"], margins=True))

    err = df[~df["exact_match"]].copy()
    print("\nErrors:", len(err), "/", len(df))
    cols = [
        "id", "source", "status",
        "gold_safety_label", "pred_safety_label",
        "gold_topic_label", "pred_topic_label",
        "gold_action", "pred_action",
        "gold_attack_type", "pred_attack_type",
        "raw_pred",
    ]
    display(err[cols].head(n))

show_error_analysis(val_pred_df, "validation")
show_error_analysis(test_pred_df, "test")


## 14. Optional: merge LoRA sang 16-bit model

Chạy khi đã hài lòng kết quả. Folder merged lớn hơn adapter.



In [ ]:

MERGE_MODEL = False

if MERGE_MODEL:
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
    print("Saved merged model to", MERGED_DIR)
else:
    print("MERGE_MODEL=False; skip merge.")


## 15. Optional: push adapter/merged model lên Hugging Face Hub

Bật sau khi đã login bằng `huggingface-cli login` hoặc có `HF_TOKEN`.



In [ ]:
PUSH_TO_HUB = False
HF_REPO_ID_ADAPTER = "your-username/qwen35-08b-guardrail-router-lora"
HF_REPO_ID_MERGED = "your-username/qwen35-08b-guardrail-router-merged"

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID_ADAPTER)
    tokenizer.push_to_hub(HF_REPO_ID_ADAPTER)
    print("Pushed adapter to", HF_REPO_ID_ADAPTER)
else:
    print("PUSH_TO_HUB=False; skip upload.")


## 16. Checklist đọc kết quả

Mốc pass v1 nên nhắm tới:

```text
valid_json_rate >= 99.5%
safety_label_macro_f1 >= 0.95
action_accuracy >= 0.95
topic_label_macro_f1 >= 0.90
unsafe_or_jailbreak_false_allow_rate <= 1-2%
```

Nếu fail:

- JSON lỗi nhiều → thêm system rule/format data, giảm LR, không tăng epoch vội.
- `AMBIGUOUS` recall thấp → oversample ambiguous hoặc thêm test/train ambiguous.
- `UNSAFE/JAILBREAK -> SAFE` cao → augment hard safety + prompt injection.
- `ON_TOPIC/OFF_TOPIC` lẫn nhau → tăng hard negative cùng course/unit.
- `selected_kp_ids` kém → v2 nên đưa `candidate_kp_ids` vào input và chỉ cho model chọn trong danh sách đó.

